In [78]:
# !pip install "unstructured[pdf]"
# !brew install poppler
# !brew install tesseract
# !pip install -qU langchain-unstructured
# !pip install langchain_qdrant
# from langchain_community.document_loaders import DirectoryLoader

In [79]:
import os
import glob
import logging
from py_modules.document_loader import DocumentLoader, UnstructuredPDFMethod
from pprint import pprint
import warnings
warnings.filterwarnings("ignore")

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_ollama import OllamaEmbeddings


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

from langchain import hub
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import fitz
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image


def plot_pdf_with_boxes(pdf_page, segments):
    pix = pdf_page.get_pixmap()
    pil_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(pil_image)
    categories = set()
    category_to_color = {
        "Title": "orchid",
        "Image": "forestgreen",
        "Table": "tomato",
    }
    for segment in segments:
        points = segment["coordinates"]["points"]
        layout_width = segment["coordinates"]["layout_width"]
        layout_height = segment["coordinates"]["layout_height"]
        scaled_points = [
            (x * pix.width / layout_width, y * pix.height / layout_height)
            for x, y in points
        ]
        box_color = category_to_color.get(segment["category"], "deepskyblue")
        categories.add(segment["category"])
        rect = patches.Polygon(
            scaled_points, linewidth=1, edgecolor=box_color, facecolor="none"
        )
        ax.add_patch(rect)

    # Make legend
    legend_handles = [patches.Patch(color="deepskyblue", label="Text")]
    for category in ["Title", "Image", "Table"]:
        if category in categories:
            legend_handles.append(
                patches.Patch(color=category_to_color[category], label=category)
            )
    ax.axis("off")
    ax.legend(handles=legend_handles, loc="upper right")
    plt.tight_layout()
    plt.show()


def render_page(doc_list: list, page_number: int, print_text=True) -> None:
    pdf_page = fitz.open(file_paths[0]).load_page(page_number - 1)
    page_docs = [
        doc for doc in doc_list if doc.metadata.get("page_number") == page_number
    ]
    segments = [doc.metadata for doc in page_docs]
    plot_pdf_with_boxes(pdf_page, segments)
    if print_text:
        for doc in page_docs:
            print(f"{doc.page_content}\n")

## File Paths

In [80]:
# Creating file paths
DATA_DIR = "./DataFiles/PDFs"
file_paths = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
print(*file_paths, sep="\n")

./DataFiles/PDFs/layout-parser-paper.pdf


## Document Loading

In [81]:
# Instantiate document loader and set parsing strategy
loader = DocumentLoader(parsing_method=UnstructuredPDFMethod())

# Read pdf files
documents = []
for file in file_paths:
    documents += loader.get_document(file_path=file)

pprint(f"Total {len(documents)} pages were read from {len(file_paths)} files.")
# pprint(documents[0])

'Total 181 pages were read from 1 files.'


In [82]:
# render_page(documents, 4)

In [83]:
# # Splitting the documents into chunks
# splitter = RecursiveCharacterTextSplitter(
#                                         chunk_size=1000,
#                                         chunk_overlap=200,
#                                         add_start_index=True
#                                         )

# documents_split = splitter.split_documents(documents)

# print(f"Total {len(documents_split)} splits were created from {len(documents)} pages.")

In [84]:
# documents_cat = [x.metadata['category'] for x in documents]
# # print(documents_cat)

# unique_categories = set(documents_cat)
# # print(unique_categories)

# # Image category documents
# image_docs = [x for x in documents if x.metadata['category'] == 'Image']
# image_docs[0].page_content

## Vector Database

In [85]:
# Initializing the Embedding model and Vector Store
embedding = OllamaEmbeddings(model="llama3.2")

client = QdrantClient(":memory:")

client.create_collection(
                        collection_name="demo_collection",
                        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
                        )


vector_store = QdrantVectorStore(
                                client=client,
                                collection_name="demo_collection",
                                embedding=embedding,
                                )


# Add the documents to the Vector Store
vector_store.add_documents(documents) #changed from documents_split to documents

# Setting up the retrievar
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6})

## Prompt

In [86]:
# Using a predefined prompt
# prompt = hub.pull("rlm/rag-prompt")

system_message = """
You are an assitant who helps in answering questions based on the context provided to you. Provide the paper title for upon which you base your response.
Answer with 'I do not know or I am not sure' if you cannot find answers in the provided context.
"""

human_message = """Based on the following context.
Context: {context}.
---
Answer the question : {question}.

In your answer qoute the provided content metadata at the end of the answer.
Answer : """

prompt = ChatPromptTemplate(
    [
        ("system", system_message),
        ("human", human_message)

    ]
)

In [87]:
# parser = StrOutputParser()

# chain = prompt 

# response = chain.invoke({"question" : "my question", "context" : "my provided context"})
# print(parser.parse(response).to_messages()[1].content)

In [88]:
# example_messages = prompt.invoke({
#                                 "context": "filler context", 
#                                 "question": "filler question"
#                                 }).to_messages()

# print(example_messages)

## LLM chain creation

In [89]:
model_name = "llama3.2"
llm = OllamaLLM(model=model_name)

In [90]:
# documents[0].metadata

In [91]:
# Function to combine the retrieved documents
def format_docs(docs):
    # print(len(docs))
    context_string = ""
    for doc in docs:
        context_string += f"""Content:- \n{doc.page_content}\n---\nMetadata:-\nFile name : {doc.metadata["filename"]} | Page number : {doc.metadata["page_number"]} | Category : {doc.metadata["category"]}\n---\n\n"""
    
    # print("="*100)
    # print(context_string)
    # print("="*100)
    
    return context_string
    # return "\n\n".join(f"{doc.page_content}" for doc in docs)


rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
            )

## QA Examples

In [92]:
logging.getLogger().setLevel(logging.CRITICAL)

In [93]:
for chunk in rag_chain.stream("Discuss the Deep learning methods for homography estimation in the papers."):
    print(chunk, end="", flush=True)

I do not know or I am not sure about discussing deep learning methods for homography estimation as it is not mentioned in the provided context. However, the paper mentions Connectionist Temporal Classiﬁcation (CTC) loss [10] which might be related to deep learning models.

But there are papers mentioned on Deep Learning Layout Annotate Layout Dataset and Training & Inference (Section 11) where they talk about layout data but does not provide details of how it uses deep learning methods for homography estimation.

In [95]:
for chunk in rag_chain.stream("Explain about the images which are availble in the provided papers?"):
    print(chunk, end="", flush=True)

Based on the provided context, it seems that there are no explicit descriptions or explanations about the images available in the paper. The text mentions "document image data" and "handling document image data", but it does not specifically describe what kind of images can be found in the paper.

However, based on the metadata provided, it is mentioned that the paper is categorized as "Image" and has a file name "layout-parser-paper.pdf". Page 11 of the paper mentions " ¥ aia Deep Leaming Layout Model Training & Inference", which suggests that there may be visual representations or diagrams related to deep learning models, but these are not explicitly described as images.

The metadata also mentions page numbers and categories, but does not provide any information about specific images. It is possible that the images are included in the paper as part of the illustrations, figures, or diagrams accompanying the text, but they are not explicitly mentioned in the content.

In summary, the

In [96]:
for chunk in rag_chain.stream("Which figures are discussed in the provided paper?"):
    print(chunk, end="", flush=True)

I do not know or I am not sure about which specific figures are discussed in the provided paper, as the context only mentions text-based layout analysis methods and various references, but does not explicitly mention any visual figures.

Metadata:
File name : layout-parser-paper.pdf | Page number : 3 | Category : NarrativeText

In [97]:
for chunk in rag_chain.stream("List the key findings in conclusion."):
    print(chunk, end="", flush=True)

Based on the context provided, I was unable to find a specific paper title or section that discusses "key findings in conclusion" directly. However, based on the overall content and metadata, it can be inferred that the key findings are related to the capabilities and features of LayoutParser, such as its DL-based CNN-RNN OCR model, simplicity, intuitiveness, and versatility.

The text mentions that LayoutParser is being actively maintained, and support for more deep learning models and novel methods in text-based layout analysis is planned (Section 5). It also highlights the convenience of using Python APIs and the ease of installation via pip. Additionally, the tool's ability to seamlessly integrate with existing DIA pipelines and its detailed documentations and tutorials are mentioned as benefits.

I do not know or I am not sure about specific key findings in conclusion, but based on the context provided, it can be inferred that LayoutParser is a powerful tool for text-based layout 

In [98]:
for chunk in rag_chain.stream("What are the mentioned approachs in the papers?"):
    print(chunk, end="", flush=True)

The approaches mentioned in the paper include:

1. Deep learning models and novel methods for text-based layout analysis (Section 5).
2. CNN-RNN OCR model trained with the Connectionist Temporal Classiﬁcation (CTC) loss.
3. Aative Learning Layout Annotate Layout Dataset
4. Deep Leaming Layout Model Training & Inference
5. Handy Data Structures & Lees Lats for Layout Data
6. LayoutParser Digitization Pipeline Modules

These approaches aim to improve the efficiency, robustness, and simplicity of document processing tasks.

Metadata:
File name : layout-parser-paper.pdf | Page number : 3 | Category : NarrativeText

In [99]:
for chunk in rag_chain.stream("Explain in detail about 'The Core LayoutParser Library'"):
    print(chunk, end="", flush=True)

Based on the provided context, the core LayoutParser library is a deep learning-based library that provides a comprehensive solution for text recognition and layout analysis tasks. The library aims to achieve precision, efficiency, and robustness in document processing tasks while being simple and lightweight.

The library provides a range of features, including:

1. Deep learning models: LayoutParser comes with a CNN-RNN OCR model trained using the Connectionist Temporal Classiﬁcation (CTC) loss. This model can be used for text recognition and layout analysis tasks.
2. Customizable: The library allows users to train their own customized datasets, making it suitable for specific application requirements.
3. Python APIs: The library provides simple and intuitive Python APIs that do not sacrifice generalizability and versatility. These APIs enable seamless integration with existing document image data pipelines.
4. Document image data handling: The library offers convenient functions for

In [100]:
for chunk in rag_chain.stream("How can LayoutParser be used for document analysis?"):
    print(chunk, end="", flush=True)

According to the provided context, LayoutParser can be used for document analysis by utilizing its deep learning-based CNN-RNN OCR model. The model can be trained on customized datasets and can be easily integrated with existing DIA pipelines.

It can also be used in sophisticated and large-scale digitization projects, and its tools such as layout structure visualization and export, text recognition, and layout annotation are beneficial for document analysis.

Additionally, LayoutParser provides simple and intuitive Python APIs without sacrificing generalizability and versatility, making it easy to use for a variety of end-users.

Metadata:
- File name: layout-parser-paper.pdf
- Page number: 2, 5, 7, 11
- Category: NarrativeText

In [101]:
for chunk in rag_chain.stream("Discuss the use cases of LayoutParser."):
    print(chunk, end="", flush=True)

LayoutParser is a library that can be used for various document processing tasks, including layout analysis and text recognition. Its use cases include:

1. Simple and light-weight document processing tasks focusing on efficiency and flexibility: LayoutParser can be easily trained on customized datasets and can handle tasks such as layout detection, text extraction, and document digitization.

2. Automatic differentiation in deep learning models: The library provides a convenient interface for automatic differentiation using the PyTorch framework.

3. Promoting reusability of layout detection models and full digitization pipelines: LayoutParser allows end-users to upload their self-trained models to a community model hub, making it easy to share and reuse models developed by others.

4. Integration with existing DIA pipelines: The library provides convenient functions for handling document image data that can be seamlessly integrated with existing Document Image Analysis (DIA) pipeline

In [103]:
for chunk in rag_chain.stream("Discuss the information which is available on Page 10."):
    print(chunk, end="", flush=True)

I do not know or I am not sure about the specific details discussed on Page 10 of the layout-parser-paper.pdf. The provided context only mentions some references and a mention of page number 10 without elaborating on what exactly is discussed on that page.

Metadata:
File name : layout-parser-paper.pdf | Page number : 10 | Category : Image